In [1]:
import tensorflow as tf
import os
import glob
import numpy as np
from model import build_1d_cnn_model
from common import INPUT_SHAPE,OUTPUT_DIM_NOTES, fast_gpu_map,parse_filtered_audio_record
cnn_model=build_1d_cnn_model(None,INPUT_SHAPE,37,False)
cnn_model.summary()
tf.keras.utils.plot_model(cnn_model,to_file='cnn_model.png',show_shapes=True)
#cnn_model.load_weights('/home/gerald/workspace/src/GuitarMidi-LV2/python/neuralnetmodelling/checkpoints/guitarmidi_epoch111_valAcc0.9598_valPrec0.7875_valRecall0.5495.h5')#
cnn_model.load_weights('guitarmidi.weights.h5')


input_filepaths = '/home/gerald/workspace/src/GuitarMidi-LV2/python/neuralnetmodelling/training_subset'#sorted(glob.glob(os.path.join(input_data_dir, '**', 'input', 'data.tfrecord'), recursive=True))
# train_dataset = tf.data.Dataset.from_tensor_slices((input_filepaths))
# train_dataset=train_dataset.shuffle(buffer_size=len(input_filepaths))
# train_dataset=train_dataset.take(100)
# train_dataset = train_dataset.map(tf_load_sample_from_files, num_parallel_calls=tf.data.AUTOTUNE)
def representative_data_gen():
    # Use TFRecordDataset to actually read the files
    # We only need a few samples to calibrate quantization
    raw_dataset = tf.data.TFRecordDataset(input_filepaths).map(parse_filtered_audio_record).take(100)
    
    # Map using your existing loading function
    calib_dataset = raw_dataset.map(lambda path: fast_gpu_map(path, training=False))
    
    for input_value, _ in calib_dataset.batch(1):
        # input_value is the (312, 256, 1) tensor
        yield [input_value]

converter = tf.lite.TFLiteConverter.from_keras_model(cnn_model)

converter.optimizations = [ tf.lite.Optimize.DEFAULT]
# converter.representative_dataset = representative_data_gen
# converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
# converter.target_spec.supported_types = [tf.int8] 

# converter.target_spec.supported_types = [tf.float16]
# converter.target_spec._experimental_supported_accumulation_type = tf.dtypes.float16
# Perform the conversion
tflite_model = converter.convert()


# converter=tf.lite.TFLiteConverter.from_keras_model(cnn_model)
# converter.optimizations = [tf.lite.Optimize.DEFAULT]
# tflite_model=converter.convert()

with open('guitarmidi.tflite','wb') as f:
    f.write(tflite_model)
print("TFLite model saved as guitarmidi.tflite")

I0000 00:00:1775820023.714994 2854920 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1775820023.737862 2854920 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1775820024.230334 2854920 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


Image height:  148


W0000 00:00:1775820024.916194 2854920 gpu_device.cc:2459] TensorFlow was not built with CUDA kernel binaries compatible with compute capability 12.0a. CUDA kernels will be jit-compiled from PTX, which could take 30 minutes or longer.
W0000 00:00:1775820024.919953 2854920 gpu_device.cc:2459] TensorFlow was not built with CUDA kernel binaries compatible with compute capability 12.0a. CUDA kernels will be jit-compiled from PTX, which could take 30 minutes or longer.
I0000 00:00:1775820024.990453 2854920 gpu_device.cc:2043] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 1302 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 5080, pci bus id: 0000:01:00.0, compute capability: 12.0a


Before string split: (None, 148, 512), max_x=148.0


Model: "guitar_note_detector"

...

 Non-trainable params: 3,984 (15.56 KB)

INFO:tensorflow:Assets written to: /tmp/tmph7i_kbmt/assets


INFO:tensorflow:Assets written to: /tmp/tmph7i_kbmt/assets


Saved artifact at '/tmp/tmph7i_kbmt'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 148, 256, 1), dtype=tf.float32, name='input_spectrogram')
Output Type:
  TensorSpec(shape=(None, 37), dtype=tf.float32, name=None)
Captures:
  135326285591376: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135326285593872: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135326285592144: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135326285593104: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135326285593296: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135326285592912: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135326285591760: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135326285594064: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135326285593680: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135326237377744: TensorSpec(shape=(), dtype=tf.resource, name=None)
  13532623

W0000 00:00:1775820031.754843 2854920 tf_tfl_flatbuffer_helpers.cc:364] Ignored output_format.
W0000 00:00:1775820031.754865 2854920 tf_tfl_flatbuffer_helpers.cc:367] Ignored drop_control_dependency.
I0000 00:00:1775820031.755112 2854920 reader.cc:83] Reading SavedModel from: /tmp/tmph7i_kbmt
I0000 00:00:1775820031.759213 2854920 reader.cc:52] Reading meta graph with tags { serve }
I0000 00:00:1775820031.759225 2854920 reader.cc:147] Reading SavedModel debug info (if present) from: /tmp/tmph7i_kbmt
I0000 00:00:1775820031.800478 2854920 mlir_graph_optimization_pass.cc:437] MLIR V1 optimization pass is not enabled
I0000 00:00:1775820031.809878 2854920 loader.cc:236] Restoring SavedModel bundle.
I0000 00:00:1775820032.100276 2854920 loader.cc:220] Running initialization op on SavedModel bundle at path: /tmp/tmph7i_kbmt
I0000 00:00:1775820032.165533 2854920 loader.cc:471] SavedModel load for tags { serve }; Status: success: OK. Took 410432 microseconds.
I0000 00:00:1775820032.235880 285492